# nb_build_import_model

Builds **NVIDIA Financials Public**, an **Import-mode** copy of the Direct Lake model **NVIDIA Financials**, so the report can use Publish to web. Same recipe as `trading-analytics-fabric/fabric/notebooks/nb_build_import_model.py`: read the Direct Lake model's BIM, swap every `directLake` partition for an Import partition on the lakehouse SQL endpoint (`Sql.Database(<endpoint>, "NVIDIALakehouse")`, schema `gold`), drop the Direct Lake shared expression and create the copy. The calculated `_Measures` table, all 84 measures, display folders and relationships come over with the BIM. The Direct Lake model and the original report are only read, never changed. No secrets.

**Kernel:** Python 3.12 (not PySpark). Attach **NVIDIALakehouse**.

**`MODE` (parameter cell)**
- `"build"` (first run): create the Import model if it doesn't exist, check structure against the source, and clone the report as **NVIDIA-financial-report - Public**. Doesn't refresh: the new model has to be mapped to an OAuth cloud connection first (see `docs/how-to-publish-public-dashboard.md`, step 3).
- `"refresh"` (after the connection is mapped, and from `pl_nvidia_sec_refresh`): sync the SQL endpoint metadata, full-refresh the Import model, then check the anchors and that all 84 measures match the Direct Lake model for FY2026.

Idempotent: an existing model or report with the target name is reused, never deleted or overwritten.

In [ ]:
%pip install -q semantic-link-labs

In [ ]:
# Parameters (the pipeline overrides MODE with "refresh")
MODE = "build"

In [ ]:
import copy
import pandas as pd
import sempy.fabric as fabric
import sempy_labs as labs

WS_ID = "d1d09fde-24c7-4984-a4b0-eb55acf7c4f4"   # NVIDIA-BalanceSheet-AI
LH_ID = "ece4bf8e-457a-4e4e-9628-4957de7a9c54"   # NVIDIALakehouse
SRC, DST = "NVIDIA Financials", "NVIDIA Financials Public"
REPORT, REPORT_PUBLIC = "NVIDIA-financial-report", "NVIDIA-financial-report - Public"
HOST, EXPECTED_MEASURES = "_Measures", 84
FY_CHECK = 2026
ANCHORS = {"Marketable Securities": 51_951_000_000, "Net Cash": 54_088_000_000}  # FY2026 10-K, USD
assert MODE in ("build", "refresh"), MODE

lh = fabric.FabricRestClient().get(f"v1/workspaces/{WS_ID}/lakehouses/{LH_ID}").json()
server = lh["properties"]["sqlEndpointProperties"]["connectionString"]
sql_endpoint_id = lh["properties"]["sqlEndpointProperties"]["id"]
db = lh["displayName"]
print("MODE:", MODE, "| SQL endpoint:", server, "| database:", db)


def dataset_id(name):
    ds = fabric.list_datasets(workspace=WS_ID)
    hit = ds[ds["Dataset Name"] == name]
    id_col = next(c for c in ds.columns if c.lower() == "dataset id")  # "Dataset Id" or "Dataset ID" by sempy version
    return None if hit.empty else hit.iloc[0][id_col]

## 1 · Build the Import copy (`MODE = "build"`)

In [ ]:
if MODE == "build":
    if dataset_id(DST):
        print(f"'{DST}' already exists: left as is (not overwritten)")
    else:
        bim = labs.get_semantic_model_bim(dataset=SRC, workspace=WS_ID)
        new = copy.deepcopy(bim)
        model = new["model"]
        changed, dl_exprs = [], set()
        for t in model["tables"]:
            for p in t.get("partitions", []):
                src = p.get("source", {})
                if p.get("mode") == "directLake" or src.get("type") == "entity":
                    schema = src.get("schemaName") or "dbo"
                    entity = src.get("entityName") or t["name"]
                    if src.get("expressionSource"):
                        dl_exprs.add(src["expressionSource"])
                    p["mode"] = "import"
                    p["source"] = {"type": "m", "expression": [
                        "let",
                        f'    Source = Sql.Database("{server}", "{db}"),',
                        f'    Data = Source{{[Schema="{schema}", Item="{entity}"]}}[Data]',
                        "in",
                        "    Data"]}
                    changed.append(f"{t['name']} <- {schema}.{entity}")
        model.pop("defaultMode", None)
        # drop the Direct Lake shared expression(s) the partitions used
        model["expressions"] = [e for e in model.get("expressions", [])
                                if e.get("name") not in dl_exprs | {"DatabaseQuery"}]
        print(len(changed), "tables switched to Import:", *changed, sep="\n  ")
        labs.create_semantic_model_from_bim(dataset=DST, bim_file=new, workspace=WS_ID)
        print("created", DST)

## 2 · Structure check: same tables, relationships, measures and folders as the source

In [ ]:
def signature(bim):
    m = bim["model"]
    j = lambda e: "\n".join(e) if isinstance(e, list) else (e or "")
    tables = sorted(t["name"] for t in m["tables"])
    measures = {(t["name"], x["name"]): (j(x.get("expression")).strip(), x.get("displayFolder", ""), x.get("formatString", ""))
                for t in m["tables"] for x in t.get("measures", [])}
    rels = sorted((r["fromTable"], r["fromColumn"], r["toTable"], r["toColumn"], r.get("isActive", True))
                  for r in m.get("relationships", []))
    return tables, measures, rels


s_t, s_m, s_r = signature(labs.get_semantic_model_bim(dataset=SRC, workspace=WS_ID))
d_bim = labs.get_semantic_model_bim(dataset=DST, workspace=WS_ID)
d_t, d_m, d_r = signature(d_bim)
dl_left = [t["name"] for t in d_bim["model"]["tables"] for p in t.get("partitions", []) if p.get("mode") == "directLake"]
on_host = [k for k in d_m if k[0] == HOST]
print(f"tables {len(d_t)}/{len(s_t)} | relationships {len(d_r)}/{len(s_r)} | measures {len(d_m)}/{len(s_m)} | on {HOST}: {len(on_host)}")

problems = []
if d_t != s_t: problems.append(f"tables differ: {sorted(set(s_t) ^ set(d_t))}")
if d_r != s_r: problems.append(f"relationships differ: {sorted(set(s_r) ^ set(d_r))}")
if d_m != s_m: problems.append(f"measures differ: {sorted({k[0][1] for k in set(s_m.items()) ^ set(d_m.items())})}")
if len(on_host) != EXPECTED_MEASURES or len(on_host) != len(d_m): problems.append(f"{len(on_host)} measures on {HOST}, expected {EXPECTED_MEASURES}")
if dl_left: problems.append(f"still Direct Lake: {dl_left}")
print("STRUCTURE CHECK:", "FAIL" if problems else "PASS")
if problems:
    raise AssertionError("; ".join(problems))

## 3 · Clone the report onto the Import model (`MODE = "build"`)

Power BI REST `Reports/Clone` with `targetModelId`, as in the trading project. The original report stays bound to the Direct Lake model.

In [ ]:
if MODE == "build":
    reports = fabric.list_reports(workspace=WS_ID)
    if (reports["Name"] == REPORT_PUBLIC).any():
        print(f"'{REPORT_PUBLIC}' already exists: left as is")
    else:
        src_report = reports[reports["Name"] == REPORT]
        assert len(src_report) == 1, f"expected one report named {REPORT}, found {len(src_report)}"
        r = fabric.PowerBIRestClient().post(
            f"v1.0/myorg/groups/{WS_ID}/reports/{src_report.iloc[0]['Id']}/Clone",
            json={"name": REPORT_PUBLIC, "targetWorkspaceId": WS_ID, "targetModelId": dataset_id(DST)})
        r.raise_for_status()
        print("cloned:", r.json().get("id"), r.json().get("name"))
    print("\nNext: map the model's data source to an OAuth 2.0 cloud connection "
          f"(server {server}, database {db}), then rerun with MODE = \"refresh\".")

## 4 · Refresh and data checks (`MODE = "refresh"`)

Syncs the SQL endpoint metadata first, so the Import refresh doesn't read a stale table right after the gold notebook writes. Then checks the anchors and that every measure returns the same FY2026 value as the Direct Lake model.

In [ ]:
def eval_all(dataset, names):
    args = ",\n    ".join(f'"{n}", CALCULATE([{n}], \'Date\'[fiscal_year] = {FY_CHECK})' for n in names)
    df = fabric.evaluate_dax(dataset=dataset, workspace=WS_ID, dax_string=f"EVALUATE ROW(\n    {args}\n)")
    row = df.iloc[0]
    return {n: row[next(c for c in df.columns if str(c) == n or str(c).endswith(f"[{n}]"))] for n in names}


def same(a, b):
    if pd.isna(a) and pd.isna(b):
        return True
    try:
        return abs(float(a) - float(b)) <= max(1e-6, 1e-9 * abs(float(b)))
    except (TypeError, ValueError):
        return str(a) == str(b)


if MODE == "refresh":
    try:
        fabric.FabricRestClient().post(f"v1/workspaces/{WS_ID}/sqlEndpoints/{sql_endpoint_id}/refreshMetadata",
                                       json={}, lro_wait=True)
        print("SQL endpoint metadata synced")
    except Exception as ex:  # sync is best-effort; the anchor checks below catch stale data
        print(f"WARN metadata sync: {type(ex).__name__}: {ex}")
    labs.refresh_semantic_model(dataset=DST, workspace=WS_ID, refresh_type="full")

    names = sorted(k[1] for k in d_m)
    got, ref = eval_all(DST, names), eval_all(SRC, names)
    failed = []
    for n, exp in ANCHORS.items():
        ok = not pd.isna(got[n]) and abs(float(got[n]) - exp) < 500_000  # anchors are in USD millions
        print(f"  {'PASS' if ok else 'FAIL'} anchor {n} FY{FY_CHECK}: {got[n]} (expected {exp:,})")
        failed += [] if ok else [n]
    diff = [n for n in names if not same(got[n], ref[n])]
    for n in diff:
        print(f"  FAIL {n}: import={got[n]} direct_lake={ref[n]}")
    print(f"  {len(names) - len(diff)}/{len(names)} measures match the Direct Lake model")
    failed += diff
    print("OVERALL:", "FAIL" if failed else "PASS")
    if failed:
        raise AssertionError("failed: " + ", ".join(failed))